# Mitigation Evaluation

Key verification, redundant privacy amplification, and TMR, evaluated against real FABRIC-collected keys and (where noted) the real classical channel.

In [ ]:
import sys, json, re, time, hashlib
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.fault_injection import SDCFaultInjector
from qne.cascade.key import key_from_sifted_json
from qne.cascade.finite_key import finite_key_output_length

print("Imports complete")


In [ ]:
SLICE_NAME = 'qfabric-bb84-2'

fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

print("Connected to slice and nodes")


## Load the real key pair, and re-sync it to both remote nodes

Explicitly re-uploading the local, known-good key files before each session avoids the exact file-divergence bug found during earlier real-channel debugging (the remote nodes' plain-named key files can silently drift from the local copy, e.g. after running `collect_key_pairs`).

In [ ]:
import json as _json

GEN_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits_genonly.json"
META_PATH = PROJECT_DIR / "results" / "fabric_key0_pe_meta.json"

if not (GEN_ALICE.exists() and GEN_BOB.exists() and META_PATH.exists()):
    raise FileNotFoundError(
        "PE-split generation-only key files not found. Run "
        "10_sdc_real.ipynb's key-loading cell first."
    )

alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
assert alice_indices == bob_indices, "alice and bob's matching indices don't match"

meta = _json.loads(META_PATH.read_text())
k_pe, real_qber = meta["k"], meta["qber"]
print(f"Loaded {alice_key.get_nr_bits()} generation bits, k={k_pe}, "
      f"real_qber (from disjoint PE sample) = {real_qber:.4f}")

# Re-sync both remote nodes to this exact (PE-split, generation-only) key pair.
alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")
print("Remote nodes re-synced to PE-split generation-only key")


## Compute the finite-key-corrected output length

Used as the privacy-amplification output length for the local mitigation tests below.

In [ ]:
n = alice_key.get_nr_bits()
k = k_pe
Q = real_qber

ell_finite, r, t, nu = finite_key_output_length(n, k, Q)
print(f"ell_finite = {ell_finite}, cascade_leakage r = {r:.1f}, verification hash t = {t:.1f}, nu = {nu:.6f}")


## Mitigation 1: Key verification (local/Mock)

First, reconcile a fault-free copy of the key locally (Mock session), to get `bob_reconciled` for the Toeplitz/hash fault tests below.

In [ ]:
session = MockClassicalSession(correct_key=alice_key)
reconciliation = Reconciliation(
    algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key,
    estimated_bit_error_rate=real_qber, seed=42,
)
bob_reconciled = reconciliation.reconcile()
assert alice_key.nr_bits_different(bob_reconciled) == 0, "Reconciliation did not converge cleanly"
print("Fault-free reconciliation complete, bob_reconciled ready")


In [ ]:
def verify_keys_match_sha256(alice_final, bob_final, verification_bits=64):
    """Lightweight post-hoc verification: compare a short SHA-256 hash of
    each party's FULL final key, rather than the key itself. NOTE: this
    hashes the entire secret key with a computationally-secure (not
    information-theoretic) hash function -- distinct from, and less
    rigorous than, the universal2-hash-of-a-sacrificed-subset verification
    now built into bob_cascade_driver.py/alice_cascade_responder.py
    (reported as 'verification_passed' in real-channel results). Kept
    separate and renamed here to avoid confusing the two."""
    alice_hash = hashlib.sha256(bytes(np.array(alice_final))).digest()[:verification_bits // 8]
    bob_hash = hashlib.sha256(bytes(np.array(bob_final))).digest()[:verification_bits // 8]
    return alice_hash == bob_hash


def run_trial_with_sha256_check(alice_key, bob_reconciled, out_len, toeplitz_prob, final_key_prob, seed,
                                   verification_bits=64):
    ext = ToeplitzHashing(input_length=alice_key.get_nr_bits(), output_length=out_len)
    injector = SDCFaultInjector(toeplitz_matrix_prob=toeplitz_prob, final_key_prob=final_key_prob, seed=seed + 2)
    pa_seed = GF2.Random(ext.seed_length)

    t0 = time.time()
    alice_final = injector.fast_toeplitz_extract_with_fault(ext, GF2(alice_key.bits), pa_seed, "alice")
    alice_final = injector.maybe_flip_final_key_bit(alice_final, "alice")
    bob_final = injector.fast_toeplitz_extract_with_fault(ext, GF2(bob_reconciled.bits), pa_seed, "bob")
    bob_final = injector.maybe_flip_final_key_bit(bob_final, "bob")
    extraction_time = time.time() - t0

    t0 = time.time()
    verified_ok = verify_keys_match_sha256(alice_final, bob_final, verification_bits)
    verification_time = time.time() - t0

    actual_match = np.array_equal(np.array(alice_final), np.array(bob_final))

    return {
        "actual_match": actual_match, "verified_ok": verified_ok,
        "true_positive": (not actual_match) and (not verified_ok),
        "false_negative": (not actual_match) and verified_ok,
        "true_negative": actual_match and verified_ok,
        "false_positive": actual_match and (not verified_ok),
        "extraction_time": extraction_time, "verification_time": verification_time,
        "faults_fired": injector.summary(),
    }


rows = []
for prob in [0.05, 0.1, 0.3, 0.5]:
    for run in range(20):
        seed = 42 + run
        result = run_trial_with_sha256_check(
            alice_key, bob_reconciled, ell_finite, toeplitz_prob=prob, final_key_prob=0.0, seed=seed)
        result["toeplitz_prob"] = prob
        result["run"] = run
        rows.append(result)

df_verification = pd.DataFrame(rows)
df_verification.to_csv(str(PROJECT_DIR / "results" / "mitigation_key_verification.csv"), index=False)

print("=== Key verification (SHA-256, full-key hash): detection performance ===")
for prob in df_verification["toeplitz_prob"].unique():
    sub = df_verification[df_verification["toeplitz_prob"] == prob]
    n_actual_mismatches = (~sub["actual_match"]).sum()
    n_caught = sub["true_positive"].sum()
    n_missed = sub["false_negative"].sum()
    detection_rate = n_caught / n_actual_mismatches if n_actual_mismatches > 0 else float('nan')
    print(f"prob={prob}: {n_actual_mismatches} real mismatches, "
          f"{n_caught} caught, {n_missed} missed -- detection rate = {detection_rate:.1%}")

print("\n=== Computational overhead ===")
print(f"Mean extraction time: {df_verification['extraction_time'].mean()*1000:.3f} ms")
print(f"Mean verification time: {df_verification['verification_time'].mean()*1000:.3f} ms")
print(f"Verification overhead: "
      f"{df_verification['verification_time'].mean() / df_verification['extraction_time'].mean() * 100:.2f}%")

n_false_positives = df_verification["false_positive"].sum()
print(f"\nFalse positives: {n_false_positives} (should be 0)")


## Real-channel reconciliation sweep, with unique output filenames

Runs 10 real-channel reconciliation trials at `reconciliation_prob=0.3`. Both driver scripts auto-suffix their output filenames with fault parameters and a timestamp, so the real path must be parsed from stdout, not assumed.

In [ ]:
def extract_output_path(stdout_text, marker):
    """Parse the actual, timestamped output filename from a script's
    printed stdout, since it's generated fresh each run."""
    match = re.search(rf"{marker} (\S+\.json)", stdout_text)
    return match.group(1) if match else None


rows = []
for run in range(10):
    seed = 42 + run
    bob_thread = bob.execute_thread(
        f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/bob_cascade_driver.py "
        f"--key-json results/bob_sifted_bits.json --alice-key-json results/alice_sifted_bits.json "
        f"--host {bob_ip} --port 5200 --qber {real_qber} --k {k_pe} --seed {seed} "
        f"--reconciliation-prob 0.3 --output results/bob_recon_run.json"
    )
    alice_thread = alice.execute_thread(
        f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/alice_cascade_responder.py "
        f"--key-json results/alice_sifted_bits.json --bob-host {bob_ip} --port 5200 "
        f"--seed {seed} --output results/alice_recon_run.json"
    )

    bob_out = bob_thread.result(timeout=180)
    try:
        alice_out = alice_thread.result(timeout=90)
    except Exception as e:
        alice_out = ("", str(e))

    bob_output_path = extract_output_path(bob_out[0], "Result written to")
    alice_output_path = extract_output_path(alice_out[0], "final key written to")

    print(f"run {run}: bob_path={bob_output_path}, alice_path={alice_output_path}")

    if bob_output_path is None:
        print(f"  WARNING: could not find Bob's output path, stderr: {bob_out[1][:300]}")
        continue

    stdout_bob_data, _ = bob.execute(f"cat ~/qfabric/{bob_output_path}", quiet=True)
    bob_data = json.loads(stdout_bob_data)

    row = {"run": run, "seed": seed,
           **{k: v for k, v in bob_data.items() if k not in ("bob_final_key", "bob_reconciled_bits")}}
    row["bob_output_path"] = bob_output_path
    row["alice_output_path"] = alice_output_path
    rows.append(row)

    time.sleep(2)  # avoid port-release race between iterations

df_unique_recon = pd.DataFrame(rows)
df_unique_recon.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_reconciliation_unique.csv"), index=False)
print(df_unique_recon.to_string(index=False))


## Mitigation 1 (continued): key verification against the exact real, saved corrupted keys

Uses the actual `bob_final_key`/`alice_final_key` produced by the real-channel runs above (not a reconstructed approximation), confirming key verification catches genuine reconciliation-fault-induced corruption, not just Toeplitz/hash faults.

In [ ]:
verification_rows = []
for _, row in df_unique_recon.iterrows():
    bob_path = row["bob_output_path"]
    alice_path = row["alice_output_path"]

    if pd.isna(bob_path) or pd.isna(alice_path):
        print(f"run {row['run']}: missing output path, skipping")
        continue

    stdout_bob, _ = bob.execute(f"cat ~/qfabric/{bob_path}", quiet=True)
    stdout_alice, _ = alice.execute(f"cat ~/qfabric/{alice_path}", quiet=True)

    bob_data = json.loads(stdout_bob)
    alice_data = json.loads(stdout_alice)

    if "bob_final_key" not in bob_data or "alice_final_key" not in alice_data:
        print(f"run {row['run']}: missing final key data, skipping")
        continue

    bob_final = np.array(bob_data["bob_final_key"])
    alice_final = np.array(alice_data["alice_final_key"])

    actual_match = np.array_equal(alice_final, bob_final)
    verified_ok = verify_keys_match_sha256(alice_final, bob_final)

    verification_rows.append({
        "run": row["run"],
        "remaining_errors_after_reconciliation": row["remaining_errors_after_reconciliation"],
        "actual_match": actual_match,
        "verified_ok": verified_ok,
        "caught": (not actual_match) and (not verified_ok),
    })

df_real_recon_verify = pd.DataFrame(verification_rows)
print(df_real_recon_verify.to_string(index=False))

n_corrupted = (~df_real_recon_verify["actual_match"]).sum()
n_caught = df_real_recon_verify["caught"].sum()
if n_corrupted > 0:
    print(f"\nReal reconciliation-fault corruption: {n_corrupted} cases, {n_caught} caught "
          f"({n_caught/n_corrupted:.1%} detection rate)")
else:
    print("\nNo corrupted cases found in this batch")

df_real_recon_verify.to_csv(str(PROJECT_DIR / "results" / "mitigation_reconciliation_verification.csv"), index=False)


## Mitigation 2: Redundant privacy amplification

Compute the same extraction twice, independently, on the same key and seed. Ground truth (`any_mismatch`) is measured directly against a clean, fault-free extraction, not inferred indirectly.

In [ ]:
ext = ToeplitzHashing(input_length=alice_key.get_nr_bits(), output_length=ell_finite)

rows = []
for prob in [0.05, 0.1, 0.3, 0.5]:
    for run in range(20):
        seed = 42 + run
        shared_seed = GF2.Random(ext.seed_length)
        clean_output = np.array(ext.extract(GF2(alice_key.bits), shared_seed))

        injector_1 = SDCFaultInjector(toeplitz_matrix_prob=prob, seed=seed + 100)
        out_1 = injector_1.fast_toeplitz_extract_with_fault(ext, GF2(alice_key.bits), shared_seed, "rep1")

        injector_2 = SDCFaultInjector(toeplitz_matrix_prob=prob, seed=seed + 200)
        out_2 = injector_2.fast_toeplitz_extract_with_fault(ext, GF2(alice_key.bits), shared_seed, "rep2")

        rep1_mismatch = not np.array_equal(out_1, clean_output)
        rep2_mismatch = not np.array_equal(out_2, clean_output)
        any_mismatch = rep1_mismatch or rep2_mismatch

        repeats_agree = np.array_equal(out_1, out_2)
        detected = any_mismatch and not repeats_agree

        rows.append({
            "prob": prob, "run": run,
            "rep1_mismatch": rep1_mismatch, "rep2_mismatch": rep2_mismatch,
            "any_mismatch": any_mismatch,
            "repeats_agree": repeats_agree, "detected": detected,
        })

df_redundant_v2 = pd.DataFrame(rows)
df_redundant_v2.to_csv(str(PROJECT_DIR / "results" / "mitigation_redundant_pa_v2.csv"), index=False)

print("=== Redundant PA: mismatch rate vs. detection rate ===")
for prob in df_redundant_v2["prob"].unique():
    sub = df_redundant_v2[df_redundant_v2["prob"] == prob]
    n_total = len(sub)
    n_mismatch = sub["any_mismatch"].sum()
    n_detected = sub["detected"].sum()
    mismatch_rate = n_mismatch / n_total
    if n_mismatch > 0:
        detection_rate = n_detected / n_mismatch
        print(f"prob={prob}: mismatch_rate={mismatch_rate:.1%} ({n_mismatch}/{n_total}), "
              f"detection_rate={detection_rate:.1%} ({n_detected}/{n_mismatch})")
    else:
        print(f"prob={prob}: mismatch_rate={mismatch_rate:.1%}, no real corruption occurred")


## Mitigation 3: Triple Modular Redundancy (TMR), real channel

Run Cascade reconciliation three times independently over the real classical channel at `reconciliation_prob=0.3`, and bitwise-majority-vote across the three replicas' outputs.

In [ ]:
def tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run, base_seed, fault_prob, k, n_replicas=3, timeout=180):
    """Run reconciliation n_replicas times over the real channel, each
    with an independent fault draw, then bitwise-majority-vote across
    the replicas' reconciled keys."""
    replica_results = []

    for rep in range(n_replicas):
        seed = base_seed + rep * 1000
        bob_output = f"results/bob_tmr_run{run}_rep{rep}.json"

        bob_thread = bob.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/bob_cascade_driver.py "
            f"--key-json results/bob_sifted_bits.json --alice-key-json results/alice_sifted_bits.json "
            f"--host {bob_ip} --port 5200 --qber {real_qber} --k {k} --seed {seed} "
            f"--reconciliation-prob {fault_prob} --output {bob_output}"
        )
        alice_thread = alice.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/alice_cascade_responder.py "
            f"--key-json results/alice_sifted_bits.json --bob-host {bob_ip} --port 5200 "
            f"--seed {seed} --output results/alice_tmr_run{run}_rep{rep}.json"
        )

        bob_out = bob_thread.result(timeout=timeout)
        try:
            alice_thread.result(timeout=timeout // 2)
        except Exception:
            pass

        bob_output_path = extract_output_path(bob_out[0], "Result written to")
        if bob_output_path is None:
            replica_results.append({"status": "crashed", "bits": None})
            continue

        stdout, _ = bob.execute(f"cat ~/qfabric/{bob_output_path}", quiet=True)
        try:
            data = json.loads(stdout)
        except json.JSONDecodeError:
            replica_results.append({"status": "no_output", "bits": None})
            continue

        if data.get("non_convergent"):
            replica_results.append({"status": "non_convergent", "bits": None})
        elif "bob_reconciled_bits" in data:
            replica_results.append({
                "status": "converged", "bits": np.array(data["bob_reconciled_bits"]),
                "remaining_errors": data.get("remaining_errors_after_reconciliation"),
            })
        else:
            replica_results.append({"status": "converged_no_bits_saved", "bits": None})

        time.sleep(2)  # avoid port-release race between replicas

    return replica_results


In [ ]:
rows = []
for run in range(5):
    base_seed = 42 + run * 10
    print(f"\n=== TMR trial {run} (real channel, prob=0.3) ===")
    replicas = tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run, base_seed, fault_prob=0.3, k=k_pe)

    statuses = [r["status"] for r in replicas]
    valid_replicas = [r["bits"] for r in replicas if r["bits"] is not None]

    print(f"  Replica statuses: {statuses}")

    if len(valid_replicas) < 2:
        rows.append({"run": run, "statuses": statuses, "result": "insufficient_valid_replicas",
                      "n_valid": len(valid_replicas), "corrected": False})
        continue

    stacked = np.stack(valid_replicas)
    majority = (stacked.sum(axis=0) > len(valid_replicas) / 2).astype(np.uint8)

    n_errors_after_tmr = alice_key.nr_bits_different(Key(bits=majority))
    corrected = n_errors_after_tmr == 0

    rows.append({"run": run, "statuses": statuses, "result": "voted",
                  "n_valid": len(valid_replicas), "n_errors_after_tmr": n_errors_after_tmr,
                  "corrected": corrected})
    print(f"  Valid replicas: {len(valid_replicas)}, errors after majority vote: {n_errors_after_tmr}, "
          f"corrected: {corrected}")

df_tmr_real = pd.DataFrame(rows)
df_tmr_real.to_csv(str(PROJECT_DIR / "results" / "mitigation_tmr_realchannel.csv"), index=False)
print("\n=== Summary ===")
print(df_tmr_real.to_string(index=False))


## Why TMR fails: error overlap analysis

Re-run one TMR trial keeping the actual bit arrays, to check whether wrong bit positions overlap across replicas (correlated faults) or are scattered independently (in which case majority voting's failure has a different, purely combinatorial explanation -- see below).

In [ ]:
def analyze_tmr_error_overlap(replicas_bits, alice_key):
    """Given a list of each replica's reconciled bit arrays, find the
    exact positions where each differs from the true key, and check
    overlap across replicas."""
    alice_arr = alice_key.bits
    error_positions = []
    for i, bits in enumerate(replicas_bits):
        wrong_positions = set(np.where(bits != alice_arr)[0].tolist())
        error_positions.append(wrong_positions)
        print(f"Replica {i}: {len(wrong_positions)} wrong bits at positions {sorted(wrong_positions)[:20]}"
              f"{'...' if len(wrong_positions) > 20 else ''}")

    for i in range(len(error_positions)):
        for j in range(i + 1, len(error_positions)):
            overlap = error_positions[i] & error_positions[j]
            union = error_positions[i] | error_positions[j]
            jaccard = len(overlap) / len(union) if union else 0
            print(f"Replica {i} vs {j}: {len(overlap)} shared wrong positions "
                  f"(Jaccard similarity = {jaccard:.2f})")

    always_wrong = set.intersection(*error_positions) if error_positions else set()
    print(f"\nPositions wrong in ALL replicas (would explain majority-vote failure): {sorted(always_wrong)}")
    return error_positions


replicas = tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run=99, base_seed=42, fault_prob=0.3, k=k_pe)
replicas_bits = [r["bits"] for r in replicas if r["bits"] is not None]

if len(replicas_bits) >= 2:
    error_positions = analyze_tmr_error_overlap(replicas_bits, alice_key)


**Finding**: overlap across replicas is low (Jaccard ~0.08-0.17), and no bit is wrong in all 3 replicas -- ruling out correlated faults as the explanation. The actual mechanism is combinatorial: with ~15 independent errors per replica scattered across ~3800 positions, some bits land wrong in *exactly* 2-of-3 replicas by chance, and majority voting confidently picks the wrong value there.

In [ ]:
alice_arr = alice_key.bits
wrong_sets = [set(np.where(bits != alice_arr)[0].tolist()) for bits in replicas_bits]

exactly_two = set()
for i, j in combinations(range(3), 2):
    pair_only = (wrong_sets[i] & wrong_sets[j])
    third = [k for k in range(3) if k != i and k != j][0]
    pair_only_not_third = pair_only - wrong_sets[third]
    exactly_two |= pair_only_not_third

print(f"Bits wrong in exactly 2 of 3 replicas: {len(exactly_two)}, positions: {sorted(exactly_two)}")
print(f"This should roughly match the observed 'errors after majority vote' count")
